In [ ]:
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
from scipy.io import loadmat

loadmat("/mnt/DATA/LEMON_Giulio/processed/lemon_aal_strin_90_ses-01.mat")[
    "subj_tcs"
].shape[0] * 1.4 / 60

In [ ]:
data = {}
for session in range(3):
    data[session] = {
        "TMI": np.load(
            f"/mnt/DATA/Giulio_NonLinearityResults/lemon_aal_strin_90_ses-0{session}_bin8/subject00_8.npy"
        )[:, 0],
        "COR": np.load(
            f"/mnt/DATA/Giulio_NonLinearityResults/lemon_aal_strin_90_ses-0{session}_bin8/subject00_8_cor.npy"
        ),
    }

In [ ]:
all_series = np.empty([6, 4005])
for session in range(3):
    all_series[2 * session, :] = data[session]["TMI"]
    all_series[2 * session + 1, :] = -0.5 * np.log10(1 - data[session]["COR"] ** 2)

In [ ]:
cov = np.corrcoef(all_series)
np.fill_diagonal(cov, np.nan)
cov.shape

In [ ]:
plt.imshow(cov)
plt.colorbar()
plt.yticks(
    range(6), labels=[f"{i+1} ses - {j}" for i in range(3) for j in ["TMI", "COR"]]
)
plt.xticks(
    range(6),
    labels=[f"{i+1} ses - {j}" for i in range(3) for j in ["TMI", "COR"]],
    rotation=90,
)
plt.show()

In [ ]:
np.nanmin(cov), np.nanmax(cov[cov < 0.97])

In [ ]:
plt.imshow(np.diff(cov, axis=0)[::2, :])
plt.colorbar().ax.set_ylabel(
    r"Corr$\langle -0.5*\log_{10}(1-\rho^2) \rangle-$Corr$\langle TMI \rangle$"
)
plt.yticks(range(3), labels=[f"{i+1} ses" for i in range(3)])
plt.xticks(
    range(6),
    labels=[f"{i+1} ses - {j}" for i in range(3) for j in ["TMI", "COR"]],
    rotation=90,
)
plt.show()
np.diff(cov, axis=0)[::2, :].shape

In [ ]:
all_sub = np.empty([3, 6, 14])
all_cov = np.empty([6, 6, 14])
for subject in range(14):
    all_series = np.empty([6, 4005])
    data = {}
    for session in range(3):
        all_series[2 * session, :] = np.load(
            f"/mnt/DATA/Giulio_NonLinearityResults/lemon_aal_strin_90_ses-0{session}_bin8/subject{subject:02}_8.npy"
        )[:, 0]
        all_series[2 * session + 1, :] = -0.5 * np.log10(
            1
            - np.load(
                f"/mnt/DATA/Giulio_NonLinearityResults/lemon_aal_strin_90_ses-0{session}_bin8/subject{subject:02}_8_cor.npy"
            )
            ** 2
        )
    cov = np.corrcoef(all_series)
    np.fill_diagonal(cov, np.nan)
    all_cov[:, :, subject] = cov
    all_sub[:, :, subject] = np.diff(cov, axis=0)[::2, :]

In [ ]:
plt.figure(figsize=(10, 10))
plt.imshow(np.mean(all_cov, axis=2))
plt.colorbar()
plt.yticks(
    range(6),
    labels=[f"{i+1} ses - {j}" for i in range(3) for j in ["TMI", r"$I_{COR}$"]],
)
plt.xticks(
    range(6),
    labels=[f"{i+1} ses - {j}" for i in range(3) for j in ["TMI", r"$I_{COR}$"]],
    rotation=90,
)
plt.title("Average across 14 subjects")
plt.show()

In [ ]:
plt.figure(figsize=(15, 10))
plt.imshow(np.mean(all_sub, axis=-1))
plt.colorbar(shrink=0.7).ax.set_ylabel(
    r"Corr$\langle -0.5*\log_{10}(1-\rho^2) \rangle-$Corr$\langle TMI \rangle$",
    fontsize=14,
)
plt.yticks(range(3), labels=[f"{i+1} ses" for i in range(3)], fontsize=15)
plt.xticks(
    range(6),
    labels=[f"{i+1} ses - {j}" for i in range(3) for j in ["TMI", r"$I_{COR}$"]],
    rotation=90,
    fontsize=15,
)
plt.ylabel("Predictors from:", fontsize=16)
plt.xlabel("Predicting:", fontsize=16)
plt.title("Average across 14 subjects", fontsize=18)
plt.show()

In [ ]:
all_sub.shape

In [ ]:
np.nanmean(all_sub[:, 0::2, :]), np.nanmean(all_sub[:, 1::2, :])

In [ ]:
plt.imshow(np.std(all_sub, axis=-1))
plt.colorbar(shrink=0.7).ax.set_ylabel(
    r"Corr$\langle -0.5*\log_{10}(1-\rho^2) \rangle-$Corr$\langle TMI \rangle$"
)
plt.yticks(range(3), labels=[f"{i+1} ses" for i in range(3)])
plt.xticks(
    range(6),
    labels=[f"{i+1} ses - {j}" for i in range(3) for j in ["TMI", "COR"]],
    rotation=90,
)
plt.ylabel("Predictors from:")
plt.xlabel("Predicting")
plt.title("Deviation across 14 subjects")
plt.show()

In [ ]:
np.where(all_sub < 0), all_sub[2, 0, 7]

In [ ]:
plt.imshow(all_sub[:, :, 7])
plt.colorbar().ax.set_ylabel(
    r"Corr$\langle -0.5*\log_{10}(1-\rho^2) \rangle-$Corr$\langle TMI \rangle$"
)
plt.yticks(range(3), labels=[f"{i+1} ses" for i in range(3)])
plt.xticks(
    range(6),
    labels=[f"{i+1} ses - {j}" for i in range(3) for j in ["TMI", "COR"]],
    rotation=90,
)
plt.show()
np.diff(cov, axis=0)[::2, :].shape

# Shadow comparison

In [ ]:
from glob import glob
import re, os, json
from matplotlib.patches import Patch

available = {}
samp_band = {1116: "delta", 2232: "theta", 3348: "alpha", 8432: "beta", 12276: "gamma"}
band_samp = {v: k for k, v in samp_band.items()}
for file in glob("/mnt/DATA/Giulio_NonLinearityResults/OLO_EEG_124s_band_*_L*_bin*"):
    mat = re.findall(r"band_([a-z]+)_L([0-9]+)_bin([0-9]+)", file)
    band, ssamp, sbin = mat[0]
    samp, bin = int(ssamp), int(sbin)
    if band not in available:
        available[band] = {}
    if samp not in available[band]:
        available[band][samp] = []
    available[band][samp].append(bin)
selected = {}
for band in available:
    if band == "gamma":
        selected["gamma"] = {
            samp_band[k]: v for k, v in available["gamma"].items() if k > 600
        }
    else:
        maxsamp = max(available[band].keys())
        selected[band] = {samp_band[maxsamp]: available[band][maxsamp]}

In [ ]:
selected

In [ ]:
RNL = {}
for band in selected:
    RNL[band] = {}
    for asband in selected[band]:
        RNL[band][asband] = {}
        for bins in selected[band][asband]:
            folder = f"/mnt/DATA/Giulio_NonLinearityResults/OLO_EEG_124s_band_{band}_L{band_samp[asband]}_bin{bins}"
            with open(
                os.path.join(folder, os.path.split(folder)[1] + "_globalStats.json")
            ) as fp:
                tmp = {k: np.array(v) for k, v in json.load(fp).items()}
            RNL[band][asband]["low" if bins < 10 else "high"] = (
                tmp["totalMIshadow"] - tmp["gaussMIshadow"]
            ) / tmp["totalMIshadow"]

In [ ]:
STRETCH = 3

basepos = np.arange(5) * STRETCH
fig = plt.figure(figsize=(15, 10))
offs = -1
for bins, colour in [("low", "blue"), ("high", "darkorange")]:
    y = [RNL[band][band][bins] for band in band_samp]
    pos = basepos + offs
    plt.boxplot(
        y,
        positions=pos,
        widths=0.6,
        boxprops={"color": colour},
        whiskerprops={"color": colour},
        flierprops={"markeredgecolor": colour, "marker": "+"},
        medianprops={"lw": 1.6, "color": colour},
        capprops={"color": colour},
    )
    offs += 0.66
for bins, colour in [("low", "darkgreen"), ("high", "goldenrod")]:
    y = [RNL["gamma"][band][bins] for band in band_samp]
    pos = basepos + offs
    plt.boxplot(
        y,
        positions=pos,
        widths=0.6,
        boxprops={"color": colour},
        whiskerprops={"color": colour},
        flierprops={"markeredgecolor": colour, "marker": "+"},
        medianprops={"lw": 1.6, "color": colour},
        capprops={"color": colour},
    )
    offs += 0.66

p1 = plt.hlines(
    0.0, basepos[0] - 0.6, basepos[-1] + 0.6, "r", "solid", lw=1, label="%", zorder=1
)
p2 = plt.hlines(
    0.01,
    basepos[0] - 0.6,
    basepos[-1] + 0.6,
    "r",
    "dotted",
    lw=1,
    label="10%",
    zorder=1,
)

plt.ylabel(f"relative difference in shadow MI")
plt.xlabel("Band")
plt.xticks(ticks=basepos, labels=band_samp.keys())
plt.legend(
    [
        Patch(facecolor="white", edgecolor="blue"),
        Patch(facecolor="white", edgecolor="darkorange"),
        Patch(facecolor="white", edgecolor="darkgreen"),
        Patch(facecolor="white", edgecolor="goldenrod"),
        p1,
        p2,
    ],
    [
        "9 bins",
        r"bins=${}^3\sqrt{samples}$",
        r"trunc. $\gamma$" + "\n9 bins",
        r"trunc. $\gamma$" + "\n" + r"bins=${}^3\sqrt{samples}$",
        r"0%",
        r"1%",
    ],
    loc=0,
)
plt.title("50 subjects $-$ Scalp Voltage $-$ SHADOW")
plt.show()